# 方案一：凸包热力学初筛


## 1. 参数配置


In [ ]:
# ============================================================
# ★★★ 用户配置区 — 修改这里即可 ★★★
# ============================================================

# --- 化学体系 ---
# 目标产物体系（用于凸包图显示 / 本征稳定性判断）
TARGET_SYSTEM = ["Fe", "Si", "S"]

# 前驱体特有元素（用于扩展体系计算）
PRECURSOR_ELEMENTS = []

# --- 凸包筛选阈值 ---
HULL_THRESHOLD = 0.5          # eV/atom，近稳定相筛选阈值

# --- MACE-MP-0 总开关 ---
MACE_ENABLED = False          # True = 启用 MACE 预测；False = 全部使用 MP 数据库数据

# --- MACE 模型与弛豫参数 ---
MACE_MODEL_PATH = "mace-mpa-0-medium.model"   # 模型文件路径/名称
MACE_DTYPE = "float64"                        # 计算精度: "float32" / "float64"
MACE_RELAX_STEPS = 1000                        # 最大迭代步数 (FIRE 达到收敛后提前终止)
M3G_FMAX = 0.10                                # 弛豫力收敛阈值 (eV/Ang)

# --- 初猜策略（策略一/二/四）---
MACE_N_RANDOM_SEEDS = 5       # 每个缺失化学式的随机初猜数量（策略一）
MACE_USE_PROTOTYPES = True    # True = 使用已知原型初猜（策略二）
MACE_VALENCE_TOL = 0.1        # 价态平衡过滤容差 (e)，越大保留候选越多
PRIMARY_ANIONS = "auto"       # "auto" = 自动检测；也可手动写 ["O","Cl"] 等
ENABLE_QUATERNARY = False     # True = 启用四元原型模板（目标明确为四元化合物时再打开）
MACE_PRECURSOR_WHITELIST = [] # 含前驱体元素的额外 MACE 候选白名单，如 ["LiMnO2"]

# --- MP 覆盖度自适应扩展 ---
AUTO_EXTEND_FROM_MP_COVERAGE = False   # 陌生体系：按 MP 覆盖度自动扩展前驱体阳离子
EXTEND_COVERAGE_THRESHOLD = 0.6       # 子系统 MP 覆盖率低于该值才扩展（0~1）
ENUM_EXTRA_CATIONS = []               # 手动指定始终纳入枚举的前驱体阳离子，如 ["Li"]

# --- 原型枚举参数 ---
MAX_ATOMS = 8                 # 每化学式的最大原子数
VOL_PER_ATOM = 18.0           # 初始体积 (Ang^3/atom)

# --- 调试模式 ---
DEBUG_MODE = False            # True = 只计算少量缺失结构，快速调试
DEBUG_N_MISSING = 3           # 调试模式下计算的缺失结构数量


## 2. 导入库与 API 设置


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP conflict
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter, PDEntry
from pymatgen.entries.mixing_scheme import MaterialsProjectDFTMixingScheme
import matplotlib.pyplot as plt

%matplotlib inline

# MACE-MP-0
try:
    from mace.calculators import mace_mp
    from ase.optimize import FIRE
    from ase import Atoms
    import torch
    _HAS_MACE = True
except ImportError:
    _HAS_MACE = False

env_loaded = load_dotenv(os.path.join("..", "api", "myapi.env"))
if not env_loaded:
    print("Warning: myapi.env not found")
    print(f"CWD: {os.getcwd()}")

API_KEY = os.getenv("MP_API_KEY")
if not API_KEY:
    raise ValueError("API Key missing")
else:
    print(f"\nAPI Key loaded (prefix: {API_KEY[:8]}...)")
    print("Ready.")

# ===== 派生参数（由上方配置自动计算，一般无需修改）=====
EXTENDED_SYSTEM = sorted(set(TARGET_SYSTEM) | set(PRECURSOR_ELEMENTS))
system = EXTENDED_SYSTEM   # 兼容旧代码：计算/导出均使用扩展体系


## 3. 获取 MP 数据与构建凸包


In [ ]:
import time

MAX_DB_RETRIES = 3
for attempt in range(1, MAX_DB_RETRIES + 1):
    try:
        with MPRester(API_KEY) as mpr:
            # 1. 获取体系数据
            entries = mpr.get_entries_in_chemsys(system, include_structure=True,
                                            additional_criteria={"thermo_types": ["GGA_GGA+U"]})
            print(f"获取到 {len(entries)} 条计算条目")

            # 2. 能量修正 + 构建凸包
            scheme = MaterialsProjectDFTMixingScheme()
            entries = scheme.process_entries(entries)
            pd = PhaseDiagram(entries)
            print(f"凸包相图构建完成 (0K, {len(entries)} 条)")
            print("   → 相图在下一步 Cell 中统一绘制（含 MACE-MP-0 对比）")
        break  # 成功则跳出重试循环
    except Exception as e:
        print(f"⚠️ 第 {attempt}/{MAX_DB_RETRIES} 次连接失败: {e}")
        if attempt < MAX_DB_RETRIES:
            wait_sec = 2 ** attempt  # 指数退避: 2s, 4s, 8s
            print(f"   {wait_sec} 秒后重试...")
            time.sleep(wait_sec)
        else:
            print(f"❌ 已重试 {MAX_DB_RETRIES} 次仍失败，请检查网络连接和 API Key")
            raise


## 4. MACE-MP-0 缺失结构能量预测


In [ ]:
# ============================================================
# MACE-MP-0 缺失结构能量预测（算法已更新为与验证版一致）
# 对 MP 数据库中不存在的候选结构，使用 MACE-MP-0 通用力场
# 弛豫并预测能量，纳入凸包热力学分析
#
# 实际项目流程：自建前驱体反应库 → 枚举候选物相 →
#   MP 已有 → 直接采用  |  MP 缺失 → MACE-MP-0 预测 → 合并凸包分析
#
# 更新内容（与 凸包计算_v2验证.ipynb 对齐）：
#   1) 原子+晶胞全弛豫 (ExpCellFilter + FIRE)
#   2) 单质参考用 is_element 查找（O2 可正确识别）
#   3) 弛豫后结构用 AseAtomsAdaptor 直接转换（不再 CIF 往返）
#   4) 增加组成一致性校验
# ============================================================

if not _HAS_MACE:
    print("MACE-MP-0 未安装，跳过。pip install mace-torch ase")
elif not MACE_ENABLED:
    print("MACE 预测已关闭（MACE_ENABLED = False），全部使用 MP 数据库数据。")
else:
    from pymatgen.entries.computed_entries import ComputedStructureEntry
    from pymatgen.core import Composition, Structure, Lattice
    from pymatgen.io.ase import AseAtomsAdaptor
    import numpy as np
    import warnings
    warnings.filterwarnings("ignore")

    # ASE 版本兼容：ExpCellFilter 在 ASE 3.22+ 位于 ase.filters
    try:
        from ase.filters import ExpCellFilter
    except ImportError:
        from ase.constraints import ExpCellFilter

    print("=" * 60)
    print("MACE-MP-0 缺失结构能量预测")
    print("=" * 60)

    # ---------- 1. 加载 MACE-MP-0 ----------
    print("\n[1/5] 加载 MACE-MP-0 通用力场...")
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"      设备: {_device}")

    # ★★★ 在这里修改您的模型文件路径 ★★★
    model_path = MACE_MODEL_PATH

    try:
        calculator = mace_mp(model=model_path, device=_device, default_dtype=MACE_DTYPE)
        print("      模型加载完成")
        _MACE_OK = True
    except Exception as _e:
        _MACE_OK = False
        print(f"      FAILED: {_e}")
        print("      Skipping MACE prediction.")

    if not _MACE_OK:
        pass
    else:

        # ---------- 1.5 计算 MP 元素参考态 + MACE 单质能量 ----------
        print("\n[1.5/5] 计算元素参考态能量 (MP & MACE)...")
        # MP 元素参考态：用 is_element 查找（O2 也能识别为 O 的单质）
        mp_elem_energy = {}  # eV/atom
        for el_symbol in system:
            el_entries = [e for e in entries
                          if e.composition.is_element and str(e.composition.elements[0]) == el_symbol]
            if el_entries:
                best_el = min(el_entries, key=lambda x: x.energy_per_atom)
                mp_elem_energy[el_symbol] = best_el.energy_per_atom
                print(f"      MP {el_symbol}: {best_el.energy_per_atom:.4f} eV/atom ({best_el.entry_id})")
            else:
                print(f"      MP {el_symbol}: ERROR - no elemental entry found!")
                mp_elem_energy[el_symbol] = None

        # MACE 单质能量：全弛豫（原子+晶胞），失败回退仅原子
        mace_elem_energy = {}
        for el_symbol in system:
            el_entries = [e for e in entries
                          if e.composition.is_element
                          and str(e.composition.elements[0]) == el_symbol
                          and hasattr(e, "structure")]
            if el_entries:
                el_struct = min(el_entries, key=lambda x: x.energy_per_atom).structure
            else:
                # 创建简单立方结构作为初猜
                atoms_init = Atoms([el_symbol], positions=[[0, 0, 0]], cell=[[5, 0, 0], [0, 5, 0], [0, 0, 5]])
                el_struct = AseAtomsAdaptor.get_structure(atoms_init)
            try:
                atoms_el = el_struct.to_ase_atoms()
                atoms_el.calc = calculator
                try:
                    opt_el = FIRE(ExpCellFilter(atoms_el), trajectory=None, logfile=None)
                    opt_el.run(fmax=M3G_FMAX, steps=MACE_RELAX_STEPS)
                    relax_tag = "原子+晶胞"
                except Exception as _ee2:
                    print(f"      MACE {el_symbol} 全弛豫失败 ({_ee2})，回退为位置弛豫")
                    opt_el = FIRE(atoms_el, trajectory=None, logfile=None)
                    opt_el.run(fmax=M3G_FMAX, steps=MACE_RELAX_STEPS)
                    relax_tag = "仅原子"
                mace_elem_energy[el_symbol] = atoms_el.get_potential_energy() / atoms_el.get_global_number_of_atoms()
                print(f"      MACE {el_symbol}: {mace_elem_energy[el_symbol]:.4f} eV/atom ({relax_tag})")
            except Exception as _ee:
                print(f"      MACE {el_symbol}: FAILED ({_ee})")
                mace_elem_energy[el_symbol] = None

        # ---------- 2. 快速验证 ----------
        print("\n[2/5] 验证 MACE-MP-0 预测精度（取 1 个 MP 条目对比）...")
        try:
            test_entry = [e for e in entries if hasattr(e, "structure") and e.structure.num_sites <= 30][0]
            test_struct = test_entry.structure
            mp_energy_pa = test_entry.energy / test_struct.num_sites

            atoms = test_struct.to_ase_atoms()
            atoms.calc = calculator
            opt = FIRE(ExpCellFilter(atoms), trajectory=None, logfile=None)
            opt.run(fmax=M3G_FMAX, steps=MACE_RELAX_STEPS)
            mace_energy_pa = atoms.get_potential_energy() / test_struct.num_sites

            delta = abs(mace_energy_pa - mp_energy_pa)
            print(f"      验证条目: {test_entry.entry_id} ({test_struct.formula})")
            print(f"      MP 能量:        {mp_energy_pa:.4f} eV/atom")
            print(f"      MACE-MP-0 能量: {mace_energy_pa:.4f} eV/atom")
            print(f"      偏差:           {delta:.4f} eV/atom")
        except Exception as _qe:
            print(f"      快速验证失败（不影响后续预测）: {_qe}")

        # ---------- 3. 扫描 MP 化学式 ----------
        print("\n[3/5] 扫描 MP 已覆盖的化学式...")
        mp_formulas = {e.composition.reduced_formula for e in entries}
        print(f"      MP 已覆盖 {len(mp_formulas)} 种化学式")

        # ---------- 4. 识别缺失项（策略一：基于已知原型的受限枚举）----------
        print("\n[4/5] 识别 MP 缺失的候选物相（原型模板枚举）...")
        from itertools import combinations, permutations, product
        from pymatgen.core import Element
        max_atoms = MAX_ATOMS
        els = [Element(s) for s in system]
        n_el = len(els)
        candidate_comps = []

        # 常见原型化学计量比模板：(名称, 元素数 k, 比例, 阴离子槽位索引)
        # X = 阴离子（O/S/N/Cl/C/F 等），模板不限于氧化物，适用于氮化物/硫化物/卤化物等
        # 二元：AB / AB2 / A2B / AB3 / A2B3
        # 三元：ABX3(钙钛矿/MX3盐) / A2BX3(M2X3?) / AB2X4(尖晶石) / ABX4 / A2B2X7(烧绿石) / ABX2(层状氧化物等)
        # 四元（ENABLE_QUATERNARY=True 时启用）：ABCX3 / ABCX4(如 LiFePO4) / A2BCX4(如 Li2FeSiO4) / A2BCX6(双钙钛矿)
        PROTOTYPE_TEMPLATES = [
            ("AB",     2, (1, 1), 1),
            ("AB2",    2, (1, 2), 1),
            ("A2B",    2, (2, 1), 1),
            ("AB3",    2, (1, 3), 1),
            ("A2B3",   2, (2, 3), 1),
            ("ABX3",   3, (1, 1, 3), 2),
            ("A2BX3",  3, (2, 1, 3), 2),
            ("AB2X4",  3, (1, 2, 4), 2),
            ("ABX4",   3, (1, 1, 4), 2),
            ("A2B2X7", 3, (2, 2, 7), 2),
            ("ABX2",   3, (1, 1, 2), 2),
            ("ABCX3",  4, (1, 1, 1, 3), 3),
            ("ABCX4",  4, (1, 1, 1, 4), 3),
            ("A2BCX4", 4, (2, 1, 1, 4), 3),
            ("A2BCX6", 4, (2, 1, 1, 6), 3),
        ]
        ANION_SYMBOLS = {"O", "S", "Se", "Te", "N", "P", "As", "C", "Si", "F", "Cl", "Br", "I", "H"}
        # 主阴离子：默认 "auto" 从目标体系+前驱体元素自动检测；也可手动指定列表
        if PRIMARY_ANIONS == "auto":
            PRIMARY_ANION_SET = {el for el in set(TARGET_SYSTEM) | set(PRECURSOR_ELEMENTS)
                                 if el in ANION_SYMBOLS}
            if not PRIMARY_ANION_SET:
                PRIMARY_ANION_SET = set(ANION_SYMBOLS)
                print("      [auto] 未检测到主阴离子，回退到全部常见阴离子集合")
            else:
                print(f"      [auto] 主阴离子: {sorted(PRIMARY_ANION_SET)}")
        else:
            PRIMARY_ANION_SET = set(PRIMARY_ANIONS)
        TARGET_SET = set(TARGET_SYSTEM)           # 目标元素（字符串）

        # 策略四：价态平衡预判（零成本过滤）
        # 检查是否存在一组常见氧化态组合使总电荷≈0；任一元素无价态数据时放行，避免误杀。
        def _is_charge_balanced(comp, tol=0.01):
            elements = list(comp.keys())
            states_list = []
            for el in elements:
                st = list(el.common_oxidation_states)
                if not st:
                    return True
                states_list.append(st)
            for states in product(*states_list):
                charge = sum(amt * s for el, amt, s in zip(elements, [comp[el] for el in elements], states))
                if abs(charge) <= tol:
                    return True
            return False

        # 泛化枚举池：只保留“目标元素 + 主阴离子”相关元素，避免前驱体无关组合爆炸
        enum_pool = [el for el in els
                     if el.symbol in TARGET_SET or el.symbol in PRIMARY_ANION_SET]

        for k in (2, 3) + ((4,) if ENABLE_QUATERNARY else ()):
            if k > len(enum_pool):
                continue
            for subset in combinations(enum_pool, k):
                # 泛化约束：至少包含一个目标元素 + 一个主阴离子
                if not any(el.symbol in TARGET_SET for el in subset):
                    continue
                if not any(el.symbol in PRIMARY_ANION_SET for el in subset):
                    continue
                # 阴离子优先选主阴离子；无主阴离子时回退到常见阴离子
                anion_cands = [el for el in subset if el.symbol in PRIMARY_ANION_SET] or \
                              [el for el in subset if el.symbol in ANION_SYMBOLS]
                if not anion_cands:
                    continue
                anion = max(anion_cands, key=lambda el: el.X)
                cations = [el for el in subset if el != anion]
                for _name, tmpl_k, ratios, anion_slot in PROTOTYPE_TEMPLATES:
                    if tmpl_k != k or sum(ratios) > max_atoms:
                        continue
                    cation_ratios = [ratios[i] for i in range(k) if i != anion_slot]
                    # 阳离子对阳离子槽位的排列都生成（交给 MP 覆盖检查/后续过滤去判断）
                    for perm in set(permutations(cations)):
                        el_map = {el.symbol: r for el, r in zip(perm, cation_ratios)}
                        el_map[anion.symbol] = ratios[anion_slot]
                        candidate_comps.append(Composition(el_map))

        # ---- 自适应扩展：前驱体阳离子子系统 MP 覆盖度检测 ----
        # 对陌生体系，若目标元素+前驱体阳离子+主阴离子的子系统在 MP 中覆盖不足，
        # 自动把该阳离子加入枚举池（只补二元 cat-阴离子 + 三元 cat-目标-阴离子）。
        def _gen_subsys_candidates(pool_els, require_cat=None):
            out = []
            for k in (2, 3):
                if k > len(pool_els):
                    continue
                for subset in combinations(pool_els, k):
                    if require_cat is not None and require_cat not in subset:
                        continue
                    # k=3 时才要求含目标元素（k=2 允许 cat+阴离子 的前驱体二元相）
                    if k == 3 and not any(el.symbol in TARGET_SET for el in subset):
                        continue
                    if not any(el.symbol in PRIMARY_ANION_SET for el in subset):
                        continue
                    anion_cands = [el for el in subset if el.symbol in PRIMARY_ANION_SET] or \
                                  [el for el in subset if el.symbol in ANION_SYMBOLS]
                    if not anion_cands:
                        continue
                    anion = max(anion_cands, key=lambda el: el.X)
                    cations = [el for el in subset if el != anion]
                    for _name, tmpl_k, ratios, anion_slot in PROTOTYPE_TEMPLATES:
                        if tmpl_k != k or sum(ratios) > max_atoms:
                            continue
                        cation_ratios = [ratios[i] for i in range(k) if i != anion_slot]
                        for perm in set(permutations(cations)):
                            el_map = {el.symbol: r for el, r in zip(perm, cation_ratios)}
                            el_map[anion.symbol] = ratios[anion_slot]
                            out.append(Composition(el_map))
            return out

        if AUTO_EXTEND_FROM_MP_COVERAGE:
            # 自动识别前驱体阳离子：在体系中、非目标元素、且不是常见阴离子（如 Li）
            auto_cats = [el for el in els
                         if el.symbol not in TARGET_SET
                         and el.symbol not in ANION_SYMBOLS
                         and el.symbol not in PRIMARY_ANION_SET]
            # 手动指定始终纳入的阳离子（即使 MP 覆盖充足）
            manual_cats = [Element(s) for s in ENUM_EXTRA_CATIONS
                           if s not in TARGET_SET and s not in PRIMARY_ANION_SET]
            for cat in list(auto_cats) + list(manual_cats):
                pool_ext = [cat] + [el for el in enum_pool if el != cat]
                ext_candidates = _gen_subsys_candidates(pool_ext, require_cat=cat)
                ext_dedup = []
                seen_ext = set()
                for c in ext_candidates:
                    rf = c.reduced_formula
                    if rf not in seen_ext:
                        seen_ext.add(rf)
                        ext_dedup.append(c)
                if not ext_dedup:
                    continue
                n_covered = sum(1 for c in ext_dedup if c.reduced_formula in mp_formulas)
                coverage = n_covered / len(ext_dedup)
                print(f"      [extend] {cat.symbol}: 候选 {len(ext_dedup)} 个，MP 覆盖 {n_covered} ({coverage:.0%})")
                is_manual = cat in manual_cats
                if is_manual or coverage < EXTEND_COVERAGE_THRESHOLD:
                    if is_manual:
                        print(f"      [extend] 手动指定，{cat.symbol} 强制加入枚举池")
                    else:
                        print(f"      [extend] MP 覆盖不足，将 {cat.symbol} 加入枚举池")
                    candidate_comps.extend(ext_dedup)

        # 前驱体白名单：即使包含前驱体元素，也显式补充为 MACE 候选（可空）
        for formula in MACE_PRECURSOR_WHITELIST:
            try:
                c = Composition(formula)
                if all(el.symbol in system for el in c.elements):
                    candidate_comps.append(c)
            except Exception:
                print(f"      [skip] 白名单化学式解析失败: {formula}")

        seen = set()
        candidate_comps_dedup = []
        for c in candidate_comps:
            rf = c.reduced_formula
            if rf not in seen:
                seen.add(rf)
                candidate_comps_dedup.append(c)
        # 应用策略四：价态平衡过滤（仅保留可能电荷平衡的组成）
        candidate_comps = [c for c in candidate_comps_dedup if _is_charge_balanced(c, tol=MACE_VALENCE_TOL)]
        print(f"      自动生成 {len(candidate_comps)} 个候选化学式（原型模板 + 主阴离子约束 + 价态平衡过滤）")

        missing_comps = []
        for comp in candidate_comps:
            rf = comp.reduced_formula
            if rf in mp_formulas:
                print(f"      [skip] {comp.formula} — MP 已覆盖")
            else:
                print(f"      [MACE] {comp.formula} — 缺失，将由 MACE-MP-0 预测")
                missing_comps.append(comp)

        if DEBUG_MODE:
            missing_comps = missing_comps[:DEBUG_N_MISSING]
            print(f"      [DEBUG] 调试模式：仅处理前 {len(missing_comps)} 个缺失结构")

        # ---------- 5. MACE-MP-0 弛豫（原子+晶胞全弛豫，多初猜取最低）----------
        # 策略一：多个随机种子初猜；策略二：已知原型初猜（perovskite/rocksalt/fluorite 等）
        print("\n[5/5] MACE-MP-0 弛豫并预测缺失结构能量...")
        mace_entries = []
        failed = []

        # ---------- 已知原型初猜生成（策略二） ----------
        def _get_prototype_structs(comp):
            """根据组成生成常见原型初猜；不支持则返回空列表。"""
            elems = sorted(comp.elements, key=lambda el: comp[el], reverse=True)
            n = len(elems)
            structs = []
            if n < 2 or n > 4:
                return structs

            def _build(proto, species):
                try:
                    a = (comp.num_atoms * VOL_PER_ATOM) ** (1 / 3)
                    if proto == "perovskite":
                        return Structure.from_prototype("perovskite", species, a=a)
                    if proto == "rocksalt":
                        return Structure.from_prototype("rocksalt", species, a=a)
                    if proto == "cscl":
                        return Structure.from_prototype("cscl", species, a=a)
                    if proto == "zincblende":
                        return Structure.from_prototype("zincblende", species, a=a)
                    if proto == "fluorite":
                        return Structure.from_prototype("fluorite", species, a=a)
                    if proto == "antifluorite":
                        return Structure.from_prototype("antifluorite", species, a=a)
                    if proto == "spinel":
                        # 尖晶石 AB2X4, Fd-3m (#227), 惯用胞 56 原子
                        a_sp = (56 * VOL_PER_ATOM) ** (1 / 3)
                        return Structure.from_spacegroup(
                            227, Lattice.cubic(a_sp), species,
                            [[0.5, 0.5, 0.5], [0.125, 0.125, 0.125], [0.26, 0.26, 0.26]])
                    if proto == "double_perovskite":
                        # 双钙钛矿 A2BCX6, Fm-3m (#225), 惯用胞 40 原子
                        a_dp = (40 * VOL_PER_ATOM) ** (1 / 3)
                        return Structure.from_spacegroup(
                            225, Lattice.cubic(a_dp), species,
                            [[0.25, 0.25, 0.25], [0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.25, 0.0, 0.0]])
                except Exception:
                    return None
                return None

            counts = sorted(comp.values())

            # 三元：ABX3 (X=阴离子) -> 钙钛矿；AB2X4 -> 尖晶石
            if n == 3:
                if counts == [1.0, 1.0, 3.0]:
                    x_el = [el for el in elems if comp[el] == 3.0][0]
                    ab = [el for el in elems if comp[el] == 1.0]
                    ab_sorted = sorted(ab, key=lambda el: el.atomic_mass, reverse=True)
                    s = _build("perovskite", ab_sorted + [x_el])
                    if s is not None:
                        structs.append(("perovskite", s))
                elif counts == [1.0, 2.0, 4.0]:
                    a_el = [el for el in elems if comp[el] == 1.0][0]
                    b_el = [el for el in elems if comp[el] == 2.0][0]
                    x_el = [el for el in elems if comp[el] == 4.0][0]
                    s = _build("spinel", [a_el, b_el, x_el])
                    if s is not None:
                        structs.append(("spinel", s))
                return structs

            # 四元：A2BCX6 -> 双钙钛矿
            if n == 4:
                if counts == [2.0, 1.0, 1.0, 6.0]:
                    a_el = [el for el in elems if comp[el] == 2.0][0]
                    b_el, c_el = [el for el in elems if comp[el] == 1.0][:2]
                    x_el = [el for el in elems if comp[el] == 6.0][0]
                    s = _build("double_perovskite", [a_el, b_el, c_el, x_el])
                    if s is not None:
                        structs.append(("double_perovskite", s))
                return structs

            # 二元
            if counts == [1.0, 1.0]:
                a_el, b_el = sorted(elems, key=lambda el: el.atomic_mass, reverse=True)
                for proto in ("rocksalt", "cscl", "zincblende"):
                    s = _build(proto, [a_el, b_el])
                    if s is not None:
                        structs.append((proto, s))
            elif counts == [1.0, 2.0]:
                # AB2 -> 萤石 (A 为 1 倍元素，B 为 2 倍元素)
                a_el = [el for el in elems if comp[el] == 1.0][0]
                b_el = [el for el in elems if comp[el] == 2.0][0]
                s = _build("fluorite", [a_el, b_el])
                if s is not None:
                    structs.append(("fluorite", s))
            elif counts == [2.0, 1.0]:
                # A2B -> 反萤石 (A 为 2 倍元素，B 为 1 倍元素)
                a_el = [el for el in elems if comp[el] == 2.0][0]
                b_el = [el for el in elems if comp[el] == 1.0][0]
                s = _build("antifluorite", [a_el, b_el])
                if s is not None:
                    structs.append(("antifluorite", s))
            return structs

        # ---------- 弛豫单个初猜 ----------
        def _relax_candidate(init_struct, comp, tag):
            """弛豫单个初猜；成功返回 dict，失败返回 None。"""
            try:
                atoms = init_struct.to_ase_atoms()
                atoms.calc = calculator
                # 单点能/受力 NaN 预检：避免异常结构让 FIRE 死锁或空转
                _e0 = atoms.get_potential_energy()
                _f0 = atoms.get_forces()
                if _e0 != _e0 or (_f0 != _f0).any():
                    print(f"         SKIP [{tag}]: 单点能/受力为 NaN")
                    return None
                print(f"         {tag}: 开始弛豫 (E0={_e0:.3f} eV)")
                opt = FIRE(ExpCellFilter(atoms), trajectory=None, logfile=None)
                opt.run(fmax=M3G_FMAX, steps=MACE_RELAX_STEPS)
                print(f"         {tag}: 弛豫结束")
                final_energy = atoms.get_potential_energy()

                final_struct = AseAtomsAdaptor.get_structure(atoms)
                actual_comp = final_struct.composition
                actual_rf = actual_comp.reduced_formula
                expected_rf = comp.reduced_formula

                if actual_rf != expected_rf:
                    print(f"         SKIP [{tag}]: composition mismatch ({actual_rf} != {expected_rf})")
                    return None

                n_atoms = actual_comp.num_atoms
                energy_pa = final_energy / n_atoms
                if abs(energy_pa) > 100:
                    print(f"         SKIP [{tag}]: energy out of range ({energy_pa:.1f} eV/atom)")
                    return None

                missing_refs = [el.symbol for el in actual_comp.keys()
                                if mace_elem_energy.get(el.symbol) is None or mp_elem_energy.get(el.symbol) is None]
                if missing_refs:
                    print(f"         SKIP [{tag}]: 元素参考缺失 {missing_refs}，无法统一坐标系")
                    return None

                corr_energy = final_energy
                for el, amt in actual_comp.items():
                    corr_energy -= amt * mace_elem_energy[el.symbol]
                    corr_energy += amt * mp_elem_energy[el.symbol]
                corr_pa = corr_energy / n_atoms
                print(f"         完成 [{tag}]: E_raw={energy_pa:.4f} eV/atom, E_corr={corr_pa:.4f} eV/atom")
                return {"raw_energy": final_energy, "corr_energy": corr_energy,
                        "corr_pa": corr_pa, "structure": final_struct, "tag": tag}
            except Exception as ex:
                print(f"         FAILED [{tag}]: {ex}")
                return None

        for comp in missing_comps:
            print(f"\n      >> {comp.formula} ...")
            num_atoms = int(sum(comp.values()))
            candidates = []

            # 策略二：已知原型初猜
            if MACE_USE_PROTOTYPES:
                for proto_name, init_struct in _get_prototype_structs(comp):
                    print(f"         原型初猜 [{proto_name}]: 生成完成")
                    cand = _relax_candidate(init_struct, comp, proto_name)
                    if cand is not None:
                        candidates.append(cand)

            # 策略一：多个随机种子初猜
            for i in range(MACE_N_RANDOM_SEEDS):
                seed = 42 + i
                rng = np.random.RandomState(seed)
                a = (num_atoms * VOL_PER_ATOM) ** (1 / 3)
                lattice = Lattice.cubic(a)
                species = []
                for el, amt in comp.items():
                    species.extend([el] * int(amt))
                coords = rng.uniform(0.15, 0.85, (num_atoms, 3))
                init_struct = Structure(lattice, species, coords)
                tag = f"seed{seed}"
                print(f"         随机初猜 [{tag}]: {num_atoms} atoms, cubic a={a:.1f} A")
                cand = _relax_candidate(init_struct, comp, tag)
                if cand is not None:
                    candidates.append(cand)

            if not candidates:
                print(f"      >> {comp.formula}: 所有初猜均失败")
                failed.append((comp.reduced_formula, "all initial guesses failed"))
                continue

            # 取校正后能量最低的候选
            best = min(candidates, key=lambda c: c["corr_pa"])
            print(f"      >> {comp.formula}: 成功 {len(candidates)} 个初猜，"
                  f"取最低 E={best['corr_pa']:.4f} eV/atom (via {best['tag']})")

            entry = ComputedStructureEntry(
                structure=best["structure"],
                energy=best["corr_energy"],
            )
            entry.data["source"] = "MACE-MP-0"
            entry.data["mace_raw_energy"] = best["raw_energy"]
            entry.data["initial_guess"] = best["tag"]
            entry.entry_id = f"MACE-{comp.reduced_formula}"
            mace_entries.append(entry)

        print(f"\n      MACE 预测成功: {len(mace_entries)} / {len(missing_comps)}")
        if failed:
            print(f"      失败/跳过 {len(failed)} 条: {failed[:5]}")
        # ---------- 6. 合并凸包 ----------
        if mace_entries:
            n_mp = len(entries)
            n_mace = len(mace_entries)
            print(f"\n{'=' * 60}")
            print(f"合并 MP ({n_mp} 条) + MACE-MP-0 ({n_mace} 条) -> 重建凸包")
            print(f"{'=' * 60}")

            entries_combined = list(entries) + mace_entries
            pd_combined = PhaseDiagram(entries_combined)

            entries_orig = entries
            pd_orig = pd
            entries = entries_combined
            pd = pd_combined

            # ---- 目标体系显示视图：仅含 TARGET_SYSTEM 元素的条目（计算/导出仍用扩展体系）----
            target_entries_orig = [e for e in entries_orig
                                   if all(el.symbol in TARGET_SYSTEM for el in e.composition.elements)]
            target_entries = [e for e in entries_combined
                              if all(el.symbol in TARGET_SYSTEM for el in e.composition.elements)]
            pd_target_orig = PhaseDiagram(target_entries_orig)
            pd_target = PhaseDiagram(target_entries)
            print(f"      扩展体系 {len(entries_combined)} 条；目标体系 {len(target_entries)} 条用于凸包显示")

            import plotly.io as pio
            pio.renderers.default = "notebook"
            print()
            print("=" * 50)
            print("MP only (目标体系, " + str(len(target_entries_orig)) + " entries)")
            print("=" * 50)
            try:
                PDPlotter(pd_target_orig, show_unstable=True).show()
            except Exception as _pe:
                print(f"      PDPlotter error (mp): {_pe}")
            print()
            print("=" * 50)
            print("MP + MACE-MP-0 (目标体系, " + str(len(target_entries)) + " entries)")
            print("=" * 50)
            try:
                PDPlotter(pd_target, show_unstable=True).show()
            except Exception as _pe:
                print(f"      PDPlotter error (combined): {_pe}")
            print()

            near_stable = [e for e in target_entries
                if pd_target.get_e_above_hull(PDEntry(e.composition, e.energy)) is not None
                and pd_target.get_e_above_hull(PDEntry(e.composition, e.energy)) < HULL_THRESHOLD]
            if near_stable:
                best = {}
                for e in near_stable:
                    rf = e.composition.reduced_formula
                    h = pd_target.get_e_above_hull(PDEntry(e.composition, e.energy))
                    if rf not in best or (h is not None and (best[rf][1] is None or h < best[rf][1])):
                        best[rf] = (e, h)
                near_stable_dedup = [v[0] for v in best.values()]
                pd_near = PhaseDiagram(near_stable_dedup)
                print("Near-stable (目标体系, " + str(len(near_stable_dedup)) + " entries):")
                try:
                    PDPlotter(pd_near, show_unstable=True).show()
                except Exception as _pe:
                    print(f"      PDPlotter error (near): {_pe}")
            else:
                print("no near-stable entries")
        else:
            print("\n所有候选结构均已在 MP 中，无需 MACE-MP-0 预测。")


In [ ]:
# ============================================================
# 导出条目供方案2使用
# 保存当前 entries (含 MP + MACE-MP-0 合并结果) 为 JSON
# 方案2 将优先从此文件加载，避免重复拉取 MP 数据
# ============================================================
import json
import os
from datetime import datetime
from monty.json import MontyEncoder

# 前置检查：确保上游 Cell 已运行
if "entries" not in dir():
    raise RuntimeError("❌ 变量 entries 未定义，请先运行前面的 Cell 5 (获取数据) 和 Cell 7 (MACE-MP-0 预测)！")

# 统计来源
n_mp = sum(1 for e in entries if getattr(e, 'data', {}).get('source', '') != 'MACE-MP-0')
n_mace = sum(1 for e in entries if getattr(e, 'data', {}).get('source', '') == 'MACE-MP-0')

# 安全序列化：三重防护处理 MP 条目 data 中的 Element 对象 key
bad_keys = 0
entry_dicts = []
for e in entries:
    try:
        entry_dicts.append(e.as_dict())
    except TypeError:
        bad_keys += 1
        # 递归将所有 dict key 转字符串
        def _fix_keys(obj):
            if isinstance(obj, dict):
                return {str(k): _fix_keys(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_fix_keys(v) for v in obj]
            return obj
        e.data = _fix_keys(e.data) if hasattr(e, 'data') and e.data else {}
        try:
            entry_dicts.append(e.as_dict())
        except TypeError:
            # 最后手段：手动构建
            d = {
                "@module": "pymatgen.entries.computed_entries",
                "@class": "ComputedStructureEntry",
                "composition": e.composition.as_dict(),
                "energy": e.energy,
                "entry_id": getattr(e, "entry_id", "unknown"),
                "correction": getattr(e, "correction", 0.0),
                "data": json.loads(json.dumps(_fix_keys(e.data), cls=MontyEncoder)),
                "structure": e.structure.as_dict() if hasattr(e, "structure") else None,
            }
            entry_dicts.append(d)

if bad_keys > 0:
    print(f"⚠️ {bad_keys} 个条目的 data 含非标准 key，已自动修复")

export_data = {
    "schema_version": 2,
    "created_at": datetime.now().isoformat(),
    "system": system,
    "target_system": TARGET_SYSTEM,
    "extended_system": EXTENDED_SYSTEM,
    "precursor_elements": PRECURSOR_ELEMENTS,
    "n_entries_mp": n_mp,
    "n_entries_mace": n_mace,
    "n_entries_total": len(entries),
    "entries": entry_dicts,
}

output_path = "scheme1_export.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, cls=MontyEncoder, indent=2)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✅ 已导出至 {output_path}")
print(f"   体系: {"-".join(system)}")
print(f"   条目: {len(entries)} 条 (MP: {n_mp}, MACE-MP-0: {n_mace})")
print(f"   文件大小: {file_size_mb:.1f} MB")
print(f"   → 方案2 可直接从此文件加载，跳过 MP API 调用")
print(f"   → 方案2 可直接从此文件加载，跳过 MP API 调用")

# ---- 目标体系显示视图（供后续 Cell 使用，仅含 TARGET_SYSTEM 元素）----
target_entries = [e for e in entries
                  if all(el.symbol in TARGET_SYSTEM for el in e.composition.elements)]
pd_target = PhaseDiagram(target_entries)
print(f"   目标体系 {TARGET_SYSTEM}: {len(target_entries)} 条用于凸包显示/结构信息")


## 5. 查看晶体结构信息


In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core import Composition, Element

# 筛选所有化合物（同时含目标体系所有元素的条目）
compounds = [e for e in target_entries
             if all(e.composition.get(Element(el), 0) > 0 for el in TARGET_SYSTEM)]

print(f"共找到 {len(compounds)} 个化合物条目：\n")
for i, e in enumerate(compounds):
    struct = e.structure
    try:
        sga = SpacegroupAnalyzer(struct, symprec=0.1)
        spg_num = sga.get_space_group_number()
        spg_symbol = sga.get_space_group_symbol()
    except Exception:
        spg_num = "?"
        spg_symbol = "无法确定"
    
    print(f"条目 {i}: {getattr(e, 'entry_id', 'N/A')}")
    pdentry = PDEntry(e.composition, e.energy)
    e_hull = pd_target.get_e_above_hull(pdentry)
    if e_hull is None:
        tag = "N/A"
    elif e_hull < 0.001:
        tag = "🟢 稳定"
    elif e_hull < HULL_THRESHOLD:
        tag = "🟡 近稳"
    else:
        tag = "🟠 亚稳"
    print(f"  稳定性: {tag} (e_hull={e_hull:.4f})")
    print(f"  化学式: {e.structure.formula}")
    print(f"  空间群: #{spg_num} {spg_symbol}")
    print(f"  能量: {e.energy:.4f} eV/atom")
    print()


## 6. 计算形成能与 e_above_hull


In [ ]:
# ============================================================
# 对所有化合物计算形成能和 e_above_hull，并写入 output.md
# ============================================================

from pymatgen.core import Composition, Element

# 筛选所有化合物
compounds = [e for e in target_entries
             if all(e.composition.get(Element(el), 0) > 0 for el in TARGET_SYSTEM)]

system_name = "-".join(TARGET_SYSTEM)

# 收集结果行，同时打印并写入 markdown
md_lines = []
md_lines.append(f"# {system_name} 体系 DFT 凸包热力学初筛结果\n")
md_lines.append(f"\n**条目总数**: {len(target_entries)} | **化合物数**: {len(compounds)}\n")
md_lines.append("\n| # | 化学式 | entry_id | 形成能 (eV/atom) | e_above_hull (eV/atom) | 稳定性 |\n")
md_lines.append("|---|--------|----------|-------------------|------------------------|--------|\n")

if compounds:
    print(f"共 {len(compounds)} 个化合物：\n")
    for idx, e in enumerate(compounds, 1):
        pdentry = PDEntry(e.composition, e.energy)
        form_e = pd_target.get_form_energy_per_atom(pdentry)
        e_above = pd_target.get_e_above_hull(pdentry)
        
        # 稳定性判断
        if e_above is None:
            stability = "N/A"
        elif e_above < 0.001:
            stability = "🟢 稳定"
        elif e_above < HULL_THRESHOLD:
            stability = "🟡 近稳"
        else:
            stability = "🟠 亚稳"
        
        # 终端打印
        print(f"🧪 {e.composition.formula}  ({getattr(e, 'entry_id', 'N/A')})")
        print(f"   🔥 形成能: {form_e:.4f} eV/atom")
        if e_above is None:
            print(f"   📊 e_above_hull: N/A")
        else:
            print(f"   📊 e_above_hull: {e_above:.4f} eV/atom  {stability}")
        print()
        
        # 追加 markdown 表格行
        e_above_str = f"{e_above:.4f}" if e_above is not None else "N/A"
        md_lines.append(f"| {idx} | {e.composition.formula} | {getattr(e, 'entry_id', 'N/A')} | {form_e:.4f} | {e_above_str} | {stability} |\n")
    
    # 写入 output.md
    with open("output.md", "w", encoding="utf-8") as f:
        f.writelines(md_lines)
    print("📄 结果已导出至 output.md")
else:
    print("❌ 未在下载的数据中找到化合物")
